## Profile a training job to identify memory bottlenecks
This notebook has a training example with un-optimized configurations and training parameters. Current notebook is tested on a single accelerator instance (g6.2xlarge). For efficient training, we are using LORA PEFT technique.
Considering the duration of workshop, we will use a trim version of `greenwich157/telco-5G-core-faults` dataset (180 records, split as train and test datasets).
The goal of this lab is to train the model and generate a nsight profile report, that can be analyzed to identify memory bottlenecks and optimized.
We will execute all cells in this notebook normally, except the last cell `lora_trainer.train()`. For the last cell, we will use Nsight Jupyterlab extension to execute and profile. Once the report is generated, open the Nsight streamer to analyze the report and observe any memory bottlenecks.
Fix as many bottlenecks as you can, and keep testing your changes by regenerating the reports. 

Reference solution with a set of memory optimization fixes are available in the notebook - 
`ft-tinyllama-lora-bf16-optimized.ipynb`

**Pre-requisites:**
- Stop any existing active Kernels, and restart this Kernel using the 'Restart Kernel' button in the toolbar.
- Enable 'Profiling with Nsight Systems' from 'NVIDIA Nsight' extension in the toolbar
- Select 'Restart Kernel' when prompted, and wait until you see a **Green** Run icon in the toolbar. Use this icon to run the last cell (`lora_trainer,train()`).

In [ ]:
!pip install datasets transformers accelerate trl peft

In [ ]:
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, IntervalStrategy
from trl import SFTTrainer

dataset = "greenwich157/telco-5G-core-faults"
model_id= "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

def prepare_training_data(dataset_id):
    data = load_dataset(dataset_id, split="train")
    data_df = data.to_pandas()
    data_df["text"] = data_df[["instruction", "input", "output"]].apply(lambda x:"<|system|>\n" + x.instruction + "</s><|user|>\n" + x.input + "</s>\n<|assistant|>\n" + x.output + "</s>", axis=1)
    data = Dataset.from_pandas(data_df)
    return data

full_train_data = prepare_training_data(dataset)
print ("training data prepared successfully ", full_train_data)
full_train_data = full_train_data.remove_columns(['instruction', 'input', 'output'])

full_train_data = full_train_data.select(range(180)) # trimming to reduce report size
full_train_data = full_train_data.train_test_split(test_size=0.20, seed=42)
print ("training data trimmed and split successfully ", full_train_data)
train_data = full_train_data['train']
test_data = full_train_data['test']

torch.cuda.empty_cache()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, dtype=torch.bfloat16)
tokenizer.pad_token = tokenizer.eos_token
#print ("tokenizer initialized successfully ", tokenizer)

lora_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="cuda",
    dtype=torch.bfloat16,
    attn_implementation="eager",
)

lora_train_args = TrainingArguments(
    bf16=True,
    bf16_full_eval=True,
    batch_eval_metrics=True,
    disable_tqdm=False,
    dataloader_num_workers=2,
    dataloader_persistent_workers=False,
    dataloader_drop_last=True,
    dataloader_pin_memory=False,
    do_eval=True,
    eval_accumulation_steps=4,
    eval_do_concat_batches=False,
    eval_strategy=IntervalStrategy.STEPS,
    full_determinism=True, 
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    logging_strategy=IntervalStrategy.STEPS,
    logging_steps=5,
    learning_rate=1e-6,
    lr_scheduler_type="constant",
    max_grad_norm=1.0,
    num_train_epochs=2,
    optim="adamw_torch",
    output_dir="ft-tinyllama-lora-bf16-baseline",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=4,
    prediction_loss_only=False,
    remove_unused_columns=False,
    skip_memory_metrics=False,
    save_strategy=IntervalStrategy.STEPS,
    save_steps=50,
    warmup_steps=0,
    warmup_ratio=0.0,
)
print ("training arguments initialized successfully ", lora_train_args)


peft_config = LoraConfig(
    lora_alpha=32, 
    lora_dropout=0.05, 
    r=16,
    bias="all",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
# print ("peft_config created ", peft_config)

lora_trainer = SFTTrainer(
    model=lora_model,
    peft_config=peft_config,
    args=lora_train_args,
    train_dataset=train_data, 
    eval_dataset=test_data,
)
print ("Trainer class initialized and staring the PEFT fine-tuning")

In [ ]:
# import torch
# print(f"GPU 0 memory: {torch.cuda.get_device_properties(0).total_memory}")
# print(f"GPU 0 allocated: {torch.cuda.memory_allocated(0)}")
# torch.cuda.empty_cache()

In [ ]:
lora_trainer.train()
print ("PEFT LORA fine-tuning Completed Successfully")

## **Guidance on report analysis and potential optimizations**

### Referrence materials
- `debug-llm-bottlenecks-with-nvidia-nsight/lab-02/02-memory-optimization/training_parameters_reference_guide.md`
- `debug-llm-bottlenecks-with-nvidia-nsight/lab-02/02-memory-optimization/ft-tinyllama-lora-bf16-optimized.ipynb`

### **Sample Observations**

![](../../static-assets/lab-02/02-memory-optimization-rep-1.png)

![](../../static-assets/lab-02/02-memory-optimization-rep-2.png)

## Please feel free to explore other sections of the report and analyze the bottlenecks, that you like to address. 

**Happy Coding !!**